# Full-Triple-Matcher (FTM) — Stage 2: Improvement 1 (Semantic Type Matching)

## Academic Project Evaluation — Semantic Meaning & Type Compatibility Integration
**Project:** Full-Triple-Matcher Enhanced Pipeline
**Base Paper:** *"Full Triple Matcher: Integrating All Triple Elements between Heterogeneous Knowledge Graphs"* (Yamamoto & Takeda, ACM Transactions on the Web, 2025)

---

### Overview of Improvement 1:
Baseline FTM and conventional KG matchers rely heavily on surface string similarity (WRatio, Levenshtein, Token Sort) and triple-level predicate similarities. However, surface similarity frequently confuses entities of completely different ontological types (e.g., a director named *Christopher Nolan* vs. a movie titled *Nolan*, or a genre *Drama* vs. a company).

**Improvement 1** introduces **Semantic Type Matching** to enforce conceptual alignment:
1. **Multi-Source Type Extraction**:
   - **Explicit RDF Types**: `rdf:type`, `wdt:P31`, `wdt:P279`, `schema:additionalType`, `dcterms:type`, `category`, `genre`.
   - **Incoming Relational Roles (Range Typing)**: Objects of `director` / `directedBy` $\to$ `director`; objects of `genre` / `category` $\to$ `genre`.
   - **Outgoing Structural Domain Signatures**: Entities having `director`, `runtime`, `releaseYear` $\to$ `movie`.
   - **URI Disambiguation Tokens**: Parsing URI suffixes e.g. `(film)`, `_movie`.
2. **Cross-Domain Synonym Clusters**: Unifies heterogeneous vocabularies (`movie` $\leftrightarrow$ `film`, `director` $\leftrightarrow$ `filmmaker`, `genre` $\leftrightarrow$ `category`).
3. **Disjoint Category Conflict Filtering**: Detects and penalizes incompatible disjoint types (e.g., `movie` vs `person`, `genre` vs `director`).
4. **Upstream Evidence for Improvements 2 & 3**:
   - In **Improvement 2**, semantic compatibility grants a **Semantic Bonus** ($-0.10 \times \text{sim}_{sem}$) that lowers the adaptive threshold for verified matching types.
   - In **Improvement 3**, semantic compatibility serves as an independent evidence term ($w_{sem} = 0.15$) in composite conflict arbitration.

In [ ]:
# 1. Setup Environment and Import Modules
import os
import sys
import json
import pandas as pd
import numpy as np

# Locate project root dynamically
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
curr = os.path.abspath(notebook_dir)
while curr and os.path.dirname(curr) != curr:
    if os.path.isdir(os.path.join(curr, "codes")) and os.path.isdir(os.path.join(curr, "dataset")):
        break
    curr = os.path.dirname(curr)
project_root = curr

entity_matching_dir = os.path.join(project_root, "codes", "entity-matching")
utils_dir = os.path.join(project_root, "codes", "utils")

for p in [project_root, entity_matching_dir, utils_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

from file_utils import load_csv, load_json, load_rdf_graph
import semantic_type_matching as stm
import adaptive_threshold as at
import conflict_resolution as cr
from entity_matching import run_pipeline
from evaluation import evaluate_predictions, build_comparison_dataframe

print("Project environment loaded successfully.")
print(f"Project root: {project_root}")

## 2. Load Knowledge Graphs and Candidate Pairs

We inspect the candidate pairs generated from the baseline preprocessing stage on the Movie dataset.

In [ ]:
kg1_path = os.path.join(project_root, "dataset", "movies", "small_movie_kg1.ttl")
kg2_path = os.path.join(project_root, "dataset", "movies", "small_movie_kg2.ttl")
cand_path = os.path.join(project_root, "dataset", "movies", "movie_candidates.csv")
gold_path = os.path.join(project_root, "dataset", "movies", "small_movie_gold.csv")

graph1 = load_rdf_graph(kg1_path)
graph2 = load_rdf_graph(kg2_path)
cand_df = load_csv(cand_path)
gold_df = load_csv(gold_path)

print(f"KG1 Triples: {len(graph1)}")
print(f"KG2 Triples: {len(graph2)}")
print(f"Candidate Pairs: {len(cand_df)}")
print(f"Gold Alignments: {len(gold_df)}")

cand_df.head(5)

## 3. Generic Semantic Type Extraction Demo

Here we demonstrate how Improvement 1 extracts semantic types from RDF graphs **generically** without hardcoding entity names.

In [ ]:
# Test entities across different categories
sample_entities = [
    ("http://kg1.example.org/resource/Inception", graph1, "KG1 Inception"),
    ("http://kg1.example.org/resource/Christopher_Nolan", graph1, "KG1 Nolan"),
    ("http://kg1.example.org/resource/Action", graph1, "KG1 Action"),
    ("http://kg2.example.org/resource/Inception_2010", graph2, "KG2 Inception 2010"),
    ("http://kg2.example.org/resource/Chris_Nolan", graph2, "KG2 Chris Nolan"),
    ("http://kg2.example.org/resource/Action_Film", graph2, "KG2 Action Film (Category)")
]

extraction_records = []
for uri, g, label in sample_entities:
    type_info = stm.extract_entity_semantic_types(g, uri)
    extraction_records.append({
        "Entity Label": label,
        "URI": uri,
        "Primary Type": type_info["primary_type"],
        "All Types": ", ".join(type_info["types"]),
        "Evidence Source": type_info["source"]
    })

pd.DataFrame(extraction_records)

## 4. Semantic Type Verification on Candidate Pairs

Now we apply `stm.apply_semantic_type_matching` across all candidates. Notice how each candidate is enriched with:
- `semantic_type_kg1`: Inferred primary category in KG1
- `semantic_type_kg2`: Inferred primary category in KG2
- `semantic_type_similarity`: Ontological similarity score in [0.0, 1.0]
- `semantic_type_match`: Compatibility boolean flag
- `semantic_type_bonus`: Bonus value passed downstream to Improvement 2
- `semantic_evidence_source`: Explainable evidence trail

In [ ]:
sem_df = stm.apply_semantic_type_matching(
    candidates_df=cand_df,
    graph1=graph1,
    graph2=graph2,
    entity_col1="e1",
    entity_col2="e2",
    sim_col="sim",
    semantic_weight=0.10,
    use_semantic_type=True
)

print(f"Total Candidates Enriched: {len(sem_df)}")
compatible_count = (sem_df["semantic_type_match"] == True).sum()
incompatible_count = (sem_df["semantic_type_match"] == False).sum()
print(f"Compatible Pairs: {compatible_count} | Incompatible Pairs: {incompatible_count}")

# Display top compatible matches
sem_df[sem_df["semantic_type_match"] == True][
    ["e1", "e2", "sim", "semantic_type_kg1", "semantic_type_kg2", "semantic_type_similarity", "semantic_type_bonus", "semantic_evidence_source"]
].head(8)

### Inspecting Incompatible Disjoint Cross-Category Pairs

Incompatible pairs (e.g. Movie vs Director or Movie vs Genre) receive `semantic_type_match = False` and `similarity = 0.0`. This prevents false positive matches.

In [ ]:
sem_df[sem_df["semantic_type_match"] == False][
    ["e1", "e2", "sim", "semantic_type_kg1", "semantic_type_kg2", "semantic_type_similarity", "semantic_type_match", "semantic_evidence_source"]
].head(8)

## 5. Mode 2 Evaluation: Fixed Threshold (0.80) + Semantic Verification

We now execute Mode 2 via `run_pipeline` and compare its Precision, Recall, and F1-score against Mode 1 (Baseline).

In [ ]:
res_base = run_pipeline("movies", mode="baseline")
res_imp1 = run_pipeline("movies", mode="improvement-1")

comp_df = pd.DataFrame([
    {
        "Mode": "Mode 1 (Baseline FTM)",
        "Precision": res_base["metrics"]["precision"],
        "Recall": res_base["metrics"]["recall"],
        "F1-Score": res_base["metrics"]["f1"],
        "True Positives": res_base["metrics"]["true_positives"],
        "False Positives": res_base["metrics"]["false_positives"],
        "Runtime (s)": res_base["metrics"]["runtime_seconds"]
    },
    {
        "Mode": "Mode 2 (Improvement 1 — Semantic Type)",
        "Precision": res_imp1["metrics"]["precision"],
        "Recall": res_imp1["metrics"]["recall"],
        "F1-Score": res_imp1["metrics"]["f1"],
        "True Positives": res_imp1["metrics"]["true_positives"],
        "False Positives": res_imp1["metrics"]["false_positives"],
        "Runtime (s)": res_imp1["metrics"]["runtime_seconds"]
    }
])

comp_df

## 6. How Improvement 1 Connects to Improvements 2 and 3

1. **Connection to Improvement 2 (Adaptive Thresholding)**:
   $$\text{Adaptive Threshold} = \theta_{base} + \text{Uncertainty Penalty} - \text{Confidence Bonus} - \mathbf{\text{SemanticBonus}}$$
   When entities belong to compatible ontological types (e.g. `movie` $\leftrightarrow$ `film`), $\text{SemanticBonus} = 0.10 \times \text{sim}_{sem}$. This lowers the required threshold, enabling the recovery of true matches with low lexical similarity (e.g. `The_Wachowskis` $\to$ `Wachowski_Sisters` and `Romance` $\to$ `Romantic_Drama`).

2. **Connection to Improvement 3 (Conflict Resolution)**:
   $$\text{Resolution Score} = w_e \cdot \text{sim}_e + w_t \cdot \text{sim}_t + w_s \cdot \text{struct} + \mathbf{w_{sem} \cdot \text{sim}_{sem}} + w_m \cdot \text{margin} - \text{penalty}$$
   When multiple candidates compete for the same source entity, semantic type compatibility acts as an independent evidence dimension ($w_{sem} = 0.15$), resolving ambiguities in favor of conceptually compatible matches and penalizing category mismatches.